In [1]:
# set seed 
import torch
import random
import numpy as np 
import transformers
import wandb


SEED = 42 # for reproduciple 
BLOCK_SIZE = 2048 # total token for each chunk/ each example in packing

def set_seed(seed = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    transformers.set_seed(seed)

set_seed(SEED)
# wandb.init(project="LLM-Learning", name="qwen-medical-cpt", resume="allow")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


# Data Preprocessing

## Load dataset

In [2]:
from datasets import load_dataset


# Dataset 1 — wikidoc (~10k samples)
wikidoc = load_dataset("medalpaca/medical_meadow_wikidoc", split="train")

# Dataset 2 — PubMedQA (~200k samples)
pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")

In [3]:
print(wikidoc)
print(pubmedqa)

Dataset({
    features: ['input', 'output', 'instruction'],
    num_rows: 10000
})
Dataset({
    features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
    num_rows: 211269
})


# Combine datasets

In [4]:
# ghép cho wikidoc (lay output)
def extract_wikidoc(example):
    text = f"{example['input']}: {example['output']}"
    return {"text": text}

# ghép cho pubmedqa (context + long_answer)
def extract_pubmed(example):
    # vi contexts la list cac doan -> join lai thanh 1 doan cach nhau bang dau cach
    contexts = " ".join(example["context"]["contexts"])
    long_answer = example["long_answer"]

    text = f"{contexts}\n{long_answer}"
    return {"text": text}



wikidoc_text = wikidoc.map(extract_wikidoc, remove_columns=wikidoc.column_names)
pubmedqa_text = pubmedqa.map(extract_pubmed, remove_columns=pubmedqa.column_names)

print(wikidoc_text)
print(wikidoc_text[0])
print("---")
print(pubmedqa_text)
print(pubmedqa_text[0])


Dataset({
    features: ['text'],
    num_rows: 10000
})
{'text': "Can you provide an overview of the lung's squamous cell carcinoma?: Squamous cell carcinoma of the lung may be classified according to the WHO histological classification system into 4 main types: papillary, clear cell, small cell, and basaloid."}
---
Dataset({
    features: ['text'],
    num_rows: 211269
})
{'text': 'Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated. The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of disease. A cross-sectional study of patients with CRS undergoing endoscopic sinus surgery was conducted. Sinus mucosal biopsies were obtained dur

In [5]:
# concat 2 cái data lại với nhau
from datasets import concatenate_datasets

combined_datasets = concatenate_datasets([wikidoc_text, pubmedqa_text])

In [6]:
print(combined_datasets)

Dataset({
    features: ['text'],
    num_rows: 221269
})


In [7]:

combined_datasets = combined_datasets.shuffle(seed=SEED)

# take small subset because it too large, take too much computation cost
# combined_datasets = combined_datasets.select(range(5000))

print(combined_datasets[0])
print(combined_datasets[1])
print(combined_datasets)

{'text': 'How can secondary prevention be employed in the management of osteoporosis?: Effective measures for the secondary prevention of osteoporosis include pharmacological therapy and also lifestyle modification.\nThe primary goal for the treatment of osteoporosis is to reduce longtime fracture risk in patients. Increasing bone mineral density (BMD) in response to the treatment is far less important than improvement of clinical aspects of osteoporosis, i.e., osteoporotic fracture. Therefore, most of the drugs efficacy is measured by the extent they improve the fracture risk instead of increasing BMD.  During the treatment, if a single fracture happens, it does not necessarily indicate treatment failure or the need to be started on an alternative treatment or patient referral to a specialist.  Calcium and vitamin D supplementation have been found to be effective in reducing the long term fracture risk, significantly. In order to suggest the people to use vitamin D and calcium supplem

## Tokenize

In [8]:
# tokenize
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

def tokenize(examples): # examples lúc này là nguyên cái batch vì batch = true
    return tokenizer(
        examples["text"], # lấy cái list chứa batch của text
        # tokenizer của hugging face tự nhận 1 list -> trả về dict của list
        truncation=False, # cắt bớt nếu vượt quá số lượng max_token
        padding=False, # đệm padding
        # -> ko vì lát dùng packing
    )

tokenized = combined_datasets.map(
    tokenize, #
    batched=True, # tokenize nhiều sample cùng lúc
    # nếu batch = false thì example là 1 sample= {"text": "..."}
    # batch = true thì là dict of a list examples = {"text": ["...", "..."]}
    batch_size=1000,
    remove_columns=["text"],
    # loại cột text ra vì ko cần, vì map thì nó là tạo ra 2 cái column mới (với cái này là input_ids, attention_mask)
)

print(tokenized)
print(tokenized[0])

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 221269
})
{'input_ids': [4340, 646, 14246, 26248, 387, 19446, 304, 279, 6240, 315, 51268, 88666, 10704, 4820, 46923, 10953, 369, 279, 14246, 26248, 315, 51268, 88666, 10704, 2924, 35349, 5729, 15069, 323, 1083, 18899, 17030, 624, 785, 6028, 5795, 369, 279, 6380, 315, 51268, 88666, 10704, 374, 311, 7949, 35404, 58804, 5214, 304, 6835, 13, 73440, 17240, 24136, 17457, 320, 33, 6076, 8, 304, 2033, 311, 279, 6380, 374, 3041, 2686, 2989, 1091, 15673, 315, 14490, 13566, 315, 51268, 88666, 10704, 11, 600, 1734, 2572, 51268, 88666, 14212, 58804, 13, 15277, 11, 1429, 315, 279, 10975, 40165, 374, 16878, 553, 279, 12818, 807, 7269, 279, 58804, 5214, 4518, 315, 7703, 425, 6076, 13, 220, 11954, 279, 6380, 11, 421, 264, 3175, 58804, 8573, 11, 432, 1558, 537, 14312, 13216, 6380, 7901, 476, 279, 1184, 311, 387, 3855, 389, 458, 10555, 6380, 476, 8720, 44780, 311, 264, 23753, 13, 220, 95004, 323, 27071, 422, 72696, 614, 1012, 1730, 31

## Packing

In [9]:
# PACKING
# ghép các sample lại với nhau để sao cho nó = đúng max token

from datasets import Dataset

def pack_tokens(examples, block_size=2048):
    # examples["input_ids"] là list of list


    # 1. concat tất cả token thành 1 list dài
    all_tokens = []
    for ids in examples["input_ids"]:
        all_tokens.extend(ids)
        # token EOS end of sentence de ngat giua cac sample
        all_tokens.append(tokenizer.eos_token_id)

    # 2. chia thành các chunks có đúng block_size
    # chunks_input_ids = []
    # chunks_attentions_mask = []

    # # it like range(0, len() - 1, 1) but here 1 is blocksize
    # for i in range(0, len(all_tokens) - block_size, block_size):
    #     chunk = all_tokens[i: i + block_size]
    #     chunks_input_ids.append(chunk) # list of list, each list is len of block-size
    #     chunks_attentions_mask.append([1] * block_size) # because al token are real token, not padding, and this mean assign 1 * blocksize

    """
    sai logic, ko con hop le, range se xet thieu last item 
    """
    

    # 2. new logic 
    usable_chunk = len(all_tokens) // block_size 
    usable_length = usable_chunk * block_size 
    dropped_token = len(all_tokens) - usable_length
   
    print(f"Total tokens: {len(all_tokens)}") 
    print(f"Usable tokens: {usable_length}")
    print(f"Dropped tokens: {dropped_token}")

    usable_tokens = all_tokens[:usable_length] 

    chunk_input_ids = [
        usable_tokens[i: i + block_size] for i in range(0, usable_length, block_size)
    ]
    
    chunk_attention_mask = [[1] * block_size for _ in chunk_input_ids]
    

    return Dataset.from_dict({
        "input_ids": chunk_input_ids,
        "attention_mask": chunk_attention_mask,
    })

# packed = tokenized.map(
#     pack_tokens,
#     batched=True,
#     batch_size=100000,
#     remove_columns=tokenized.column_names,
# )

"""
ko sử dụng map bằng batch, vì lý do là nếu chia batch, dữ liệu sẽ bị phân chia ra thành các batch, vì thế lúc này logic gom toàn bộ sẽ ko đúng nữa
thay vào đó nó sẽ gom theo cụm batch
"""

# new logic 
packed = pack_tokens(tokenized, block_size=BLOCK_SIZE)


print(packed)
print(packed[0]["input_ids"][:10])
print(len(packed[0]["input_ids"]))
print(len(packed[0]["attention_mask"]))

Total tokens: 83318218
Usable tokens: 83316736
Dropped tokens: 1482
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 40682
})
[4340, 646, 14246, 26248, 387, 19446, 304, 279, 6240, 315]
2048
2048


# split train test 

In [10]:
split = packed.train_test_split(test_size=0.05, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

## Data Collator

In [12]:
# tạo labels từ input_ids, nó thục hiện shift 1 token sang trái
# input_ids: [50, 98, 1234, 567, ...]
# labels:    [98, 1234, 567, ???, ...] shift 1 token sang trái

from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, # mục đích để lấy tokenizer.pad_token_id
    # thục ra sau khi packing thì ko cần padding, nhưng đó là tham số bắt buộc nên vẫn phải truyền vào
    mlm=False, # causal LM, Không phải masked LM
)

# sự khác nhau giữa causal LM (GPT style) và masked LM (BERT style)
# masked che ngẫu nhiên, xong có thể nhin trái hoặc phải để predit , loss tính trên tokn bị mask
# causal là che hết phải, chỉ nhìn trái predict phải, loss ính trên toàn bộ token


Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7b95fcf25f40>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7b92e7277620, raw_cell="# tạo labels từ input_ids, nó thục hiện shift 1 to.." transformed_cell="# tạo labels từ input_ids, nó thục hiện shift 1 to.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a223130302e3131302e3136352e3430227d/home/pc5070ti/workspace/LLM-Learning/CPT.ipynb#X63sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7b95fcf25f40>> (for post_run_cell), with arguments args (<ExecutionResult object at 7b92e7274a70, execution_count=12 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7b92e7277620, raw_cell="# tạo labels từ input_ids, nó thục hiện shift 1 to.." transformed_cell="# tạo labels từ input_ids, nó thục hiện shift 1 to.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a223130302e3131302e3136352e3430227d/home/pc5070ti/workspace/LLM-Learning/CPT.ipynb#X63sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

## load Model  

In [19]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B",
    torch_dtype=torch.bfloat16, # bfloat giu nguyen phan exponent cuar float32 chi thay doi man mantissa (precision) xuong chinh xac 4 chu so thay vi 8 chu so cuar float32, trach viec gradient qua lon gay Nan
)

total_params = sum(p.numel() for p in model.parameters())
print(f"total params: {total_params}")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

total params: 494032768


## TrainingArguments

In [20]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="qwen-medical-pretrained",

    # batch_size
    per_device_train_batch_size=1, # 1 batch = 4 chunks
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32, #   1 step = 32 batches
    eval_accumulation_steps=8, # tích lũy trong gpu rồi đưa sanng cpu 
    # bình thường thì hết 1 batch là update, còn cái này nó sẽ tích lũy lại rồi mới up

    # eval 
    eval_strategy="steps",
    eval_steps=100,

    # learning rate
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=60, # warm up trong 60 steps (tương đương 5% total training steps)


    # Duration
    num_train_epochs=1,

    # Save checkpoint
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_steps=100, # steps số lần model update weight
    save_total_limit=3,
    

    # logging
    logging_steps=5,

    # Precision
    bf16=True,

    # Memory
    gradient_checkpointing=True,
    report_to="wandb",
    # run_name="qwen-medical-cpt",  # tên run trên dashboard

)

## Trainer

In [11]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)


trainer.train()

NameError: name 'model' is not defined

In [ ]:
# save
trainer.save_model("qwen-medical-pretrained")
tokenizer.save_pretrained("qwen-medical-pretrained")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('qwen-medical-pretrained/tokenizer_config.json',
 'qwen-medical-pretrained/chat_template.jinja',
 'qwen-medical-pretrained/tokenizer.json')

# Eval

In [16]:
import math
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling

# 1. Tải tokenizer và mô hình đã lưu sau CPT
tokenizer = AutoTokenizer.from_pretrained("qwen-medical-pretrained")
model = AutoModelForCausalLM.from_pretrained(
    "qwen-medical-pretrained",
    torch_dtype=torch.bfloat16
).cuda()

# 2. Tạo DataLoader từ tập dữ liệu eval_dataset
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
eval_dataloader = DataLoader(eval_dataset, batch_size=2, collate_fn=data_collator)

# 3. Chuyển model sang chế độ eval (đánh giá)
model.eval()
total_loss = 0.0
total_steps = 0

print("Đang đánh giá mô hình trên tập eval_dataset...")
with torch.no_grad(): # Tắt tính toán gradient để tiết kiệm bộ nhớ và chạy nhanh hơn
    for batch in eval_dataloader:
        # Đưa các tensor lên GPU cùng với model
        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        labels = batch["labels"].to(model.device)
        
        # Forward pass để tính loss
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        total_loss += loss.item()
        total_steps += 1

# 4. Tính toán Loss trung bình và Perplexity
mean_eval_loss = total_loss / total_steps
perplexity = math.exp(mean_eval_loss)

print("-" * 50)
print(f"Evaluation Loss (Trung bình): {mean_eval_loss:.4f}")
print(f"Perplexity (PPL): {perplexity:.4f}")
print("-" * 50)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Đang đánh giá mô hình trên tập eval_dataset...
--------------------------------------------------
Evaluation Loss (Trung bình): 1.9793
Perplexity (PPL): 7.2378
--------------------------------------------------


# Test Model 

In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch 

# 1. Định nghĩa các câu hỏi kiểm tra
medical_prompts = [
    "What are the symptoms of diabetes?",
    "How to treat hypertension?",
    "Describe the pathophysiology of heart failure."
]

# 2. Tải mô hình nguyên bản (Zero-shot)
print("Đang load mô hình ZERO-SHOT (Qwen2.5-0.5B)...")
zero_shot_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
zero_shot_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B",  
    torch_dtype=torch.bfloat16,
).cuda()

# 3. Tải mô hình đã được huấn luyện CPT
print("Đang load mô hình CPT (Medical Pretrained)...")
cpt_tokenizer = AutoTokenizer.from_pretrained("qwen-medical-pretrained")
cpt_model = AutoModelForCausalLM.from_pretrained(
    "qwen-medical-pretrained",
    torch_dtype=torch.bfloat16,
).cuda()

# 4. Hàm sinh văn bản kiểm soát lặp từ
def generate_response(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    output_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=150,             # Giới hạn số token được sinh thêm
        do_sample=True,                 # Kích hoạt sinh văn bản ngẫu nhiên có lấy mẫu
        temperature=0.7,                
        top_p=0.9,                      
        repetition_penalty=1.1,         # Tránh lặp lại một từ/cụm từ nhiều lần
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
    )
    
    # Chỉ lấy phần văn bản mới sinh, không lấy lại phần prompt đầu vào
    prompt_length = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][prompt_length:]
    
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# 5. Duyệt qua từng prompt để hiển thị so sánh
for prompt in medical_prompts: 
    print("\n" + "="*80)
    print(f"PROMPT: {prompt}")
    print("="*80)
    
    response_zero = generate_response(zero_shot_model, zero_shot_tokenizer, prompt)
    print(f"\n[ZERO-SHOT RESPONSE]:\n{response_zero}")
    
    response_cpt = generate_response(cpt_model, cpt_tokenizer, prompt)
    print(f"\n[CPT RESPONSE]:\n{response_cpt}")


Đang load mô hình ZERO-SHOT (Qwen2.5-0.5B)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Đang load mô hình CPT (Medical Pretrained)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


PROMPT: What are the symptoms of diabetes?

[ZERO-SHOT RESPONSE]:
 The symptoms can vary from person to person, but here are some common ones:

1. Frequent urination - feeling like you need to go every few hours.
2. Increased thirst
3. Increased hunger or weight loss without trying
4. Fatigue and weakness
5. Blurred vision

It's important to keep in mind that these symptoms may also be caused by other conditions, so it's always best to get checked out by a doctor if you experience any of them.

Is it normal for someone with diabetes to have night sweats? Yes, it is quite common for people with diabetes to experience night sweats. This symptom occurs when the body's temperature rises due to the increased blood flow to the skin, which can happen when

[CPT RESPONSE]:
 Diabetic patients present with a wide spectrum of clinical manifestations. The following symptoms can be indicative of diabetes, but should not necessarily be associated with diabetes:
Diabetes is usually diagnosed by medi

### Đánh giá và so sánh kết quả giữa Zero-Shot và CPT (Continue Pretraining)

Sau khi áp dụng các tham số sinh văn bản cải tiến (kích hoạt `do_sample=True` và áp dụng `repetition_penalty=1.1`), kết quả sinh ra đã không còn hiện tượng lặp từ vô hạn. Dưới đây là phân tích chi tiết cho từng prompt:

#### 1. Prompt 1: "What are the symptoms of diabetes?"
- **Zero-Shot Response**: Liệt kê được 5 triệu chứng cơ bản khá tốt. Tuy nhiên, do là mô hình chat đa dụng chung, nó có xu hướng trò chuyện tự nhiên ("The symptoms can vary...") và tự động đặt thêm câu hỏi phụ ở cuối ("Is it normal for someone with diabetes to have night sweats?...") rồi tự trả lời dở dang.
- **CPT Response**: Trả lời mang tính học thuật cao. Mô tả các triệu chứng lâm sàng và đi sâu vào phương pháp chẩn đoán chuyên môn y khoa (như đo Fasting Plasma Glucose - FPG, xét nghiệm chỉ số HbA1c). Văn phong giống như sách giáo khoa y khoa.

#### 2. Prompt 2: "How to treat hypertension?"
- **Zero-Shot Response**: Bị lỗi ảo giác (hallucination) nghiêm trọng. Mô hình tự chuyển sang dạng câu hỏi trắc nghiệm y khoa, rồi tự sinh tiếp một câu hỏi trắc nghiệm hoàn toàn lạc đề về kỹ thuật điện ("secondary circuit") và quản lý giao thông. Đây là hành vi điển hình của một Base Model chưa qua căn chỉnh hành vi (SFT).
- **CPT Response**: Đưa ra các số liệu dịch tễ học y khoa mang tính nghiên cứu khoa học (số ca mắc bệnh trên thế giới và tại Trung Quốc) và hướng nghiên cứu trên mô hình chuột spontaneously hypertensive rats (SHR). Dữ liệu có độ tin cậy chuyên môn cao nhưng mang tính học thuật nghiên cứu hơn là hướng dẫn điều trị trực tiếp.

#### 3. Prompt 3: "Describe the pathophysiology of heart failure."
- **Zero-Shot Response**: Mô tả khái quát khá ổn nhưng phần giải thích cơ chế bệnh sinh tương đối nông và ngắn, chủ yếu xoay quanh thiếu oxy cơ tim.
- **CPT Response**: Trình bày cực kỳ xuất sắc và chuyên sâu về mặt y học. Liệt kê chính xác các cơ chế bệnh lý (ventricular remodeling - tái cấu trúc thất, sympathetic activity - hoạt động giao cảm, alterations in myocardial contractility - thay đổi lực co bóp cơ tim) và phân loại rõ ràng các dạng suy tim (systolic, diastolic, mixed) cùng nguyên nhân tương ứng.

#### Kết luận chung về hiệu quả của Continue Pretraining (CPT):
1. **Tích lũy tri thức chuyên ngành**: Mô hình CPT đã hấp thụ thành công lượng lớn kiến thức y học chuyên sâu và thuật ngữ chuyên ngành (như HbA1c, ventricular remodeling, Spontaneously Hypertensive Rats) từ tập dữ liệu y khoa.
2. **Hạn chế của mô hình chỉ mới CPT**: CPT giúp mô hình có kiến thức (Knowledge) nhưng chưa định hình được hành vi phản hồi (Behavior). Mô hình có xu hướng viết tiếp văn bản y khoa như viết sách/báo cáo khoa học chứ chưa biết đóng vai trợ lý để trả lời câu hỏi trực tiếp của người dùng. Đây là lý do vì sao chúng ta cần bước tiếp theo là **SFT (Supervised Fine-Tuning)** để dạy mô hình cách tương tác và trả lời câu hỏi một cách chuẩn mực.